# QE vs QE comparison plotting (ex. pp-type, soc)

#### モジュール

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import ipynbname
NB_NAME = ipynbname.name()

#### データセットの定義

In [ ]:
# 公開用サンプル

dataset_material_a_nc_vs_paw = {
    "title": "MaterialA: QE band comparison (NC vs PAW) (example)",
    "band1": {
        "file": "./../materials/MaterialA/NC/SOC/bands/bands.out.gnu",
        "label": "NC",
        "shift": 0.0000,  # ダミー値
    },
    "band2": {
        "file": "./../materials/MaterialA/PAW/SOC/band_data/bands.out.gnu",
        "label": "PAW",
        "shift": 0.0000,  # ダミー値
    },
    "kpath": {
        "ticks":  [0, 10, 20, 30, 40, 50, 60, 70, 80, 90],
        "labels": ["Γ", "X", "S", "Y", "Γ", "Z", "U", "R", "T", "Z"],
    },
}

#### バンドファイル読み込み関数

In [ ]:
# 空行区切りのバンドファイル読み取り（QEの.gnuファイル用）
def read_separated_band_file(filename):
    bands_list = []
    current_band = []
    with open(filename) as f:
        for line in f:
            if line.strip() == "":
                if current_band:
                    bands_list.append(np.array(current_band))
                    current_band = []
            else:
                vals = line.split()
                current_band.append((float(vals[0]), float(vals[1])))
        if current_band:  # ファイル末尾に空行がない場合の対処
            bands_list.append(np.array(current_band))
    return bands_list

# バンドのリストをNaN区切りで1本の配列に結合する（plot用: 線が繋がらないようにする）
def concat_bands_with_nan(bands_list):
    pieces = []
    for band in bands_list:
        pieces.append(band)
        pieces.append(np.full((1, band.shape[1]), np.nan))
    return np.concatenate(pieces[:-1], axis=0)  # 末尾の余分なNaN行は除く

#### プロット

In [ ]:
# データセットの指定
dataset = dataset_material_a_nc_vs_paw
title = dataset["title"]

# サブプロットフレームの作成
fig, ax1 = plt.subplots(1, 1, figsize=(8, 6.0))
fig.canvas.header_visible = False

# --- band1の描画 ---
band1_list = read_separated_band_file(dataset["band1"]["file"])
band1_concat = concat_bands_with_nan(band1_list)  # バンド間が繋がらないようにNaN区切りで1本化
ax1.plot(band1_concat[:, 0], band1_concat[:, 1] - dataset["band1"]["shift"],
          color='black', linewidth=1, linestyle='-', label=dataset["band1"]["label"])

# --- band2の描画 ---
band2_list = read_separated_band_file(dataset["band2"]["file"])
band2_concat = concat_bands_with_nan(band2_list)
ax1.plot(band2_concat[:, 0], band2_concat[:, 1] - dataset["band2"]["shift"],
          color='red', linewidth=1, linestyle='--', label=dataset["band2"]["label"])

# --- 高対称点ラベルの設定 ---
kmax = max(band1_concat[:, 0].max(), band2_concat[:, 0].max())
ax1.set_xlabel("K-point")
ax1.set_ylabel("Energy (eV)")
ax1.set_xticks(dataset["kpath"]["ticks"])
ax1.set_xticklabels(dataset["kpath"]["labels"])
ax1.set_xlim(0, kmax)
ax1.legend(fontsize=10)
ax1.grid(True, linestyle=':')

fig.suptitle(title, fontsize=16)

# 画像として保存
save_dir = f"./{NB_NAME}_save"
os.makedirs(save_dir, exist_ok=True)
fig.savefig(f"{save_dir}/{re.sub(r'[^\w\-]+', '_', title)}.png", dpi=300, bbox_inches="tight")